In [ ]:
from google.colab import files
uploaded = files.upload()

Saving braille detection.cocoyolo.zip to braille detection.cocoyolo.zip


In [ ]:
import os
for name, data in uploaded.items():
    print(name, len(data), "bytes")

braille detection.cocoyolo.zip 485457069 bytes


In [ ]:
import os, zipfile

zip_candidates = [f for f in os.listdir(".") if f.endswith(".zip")]
assert len(zip_candidates) == 1, f"Expected 1 zip file, found: {zip_candidates}"
zip_path = zip_candidates[0]
print("Using zip:", zip_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("braille_coco")

print(os.listdir("braille_coco"))

Using zip: braille detection.cocoyolo.zip
['train', 'valid', 'test', 'README.dataset.txt', 'README.roboflow.txt']


In [ ]:
import json, hashlib
from PIL import Image
from collections import Counter

def load_coco(split_dir):
    with open(f"{split_dir}/_annotations.coco.json") as f:
        return json.load(f)

def check_images(split_dir):
    bad_files = []
    for fname in os.listdir(split_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            path = os.path.join(split_dir, fname)
            try:
                Image.open(path).verify()
            except Exception as e:
                bad_files.append((path, str(e)))
    return bad_files

def check_boxes(coco):
    img_dims = {img["id"]: (img["width"], img["height"]) for img in coco["images"]}
    issues = []
    for ann in coco["annotations"]:
        x, y, w, h = ann["bbox"]
        img_w, img_h = img_dims[ann["image_id"]]
        if w <= 0 or h <= 0 or x < 0 or y < 0 or x + w > img_w or y + h > img_h:
            issues.append(ann["id"])
    return issues

def class_distribution(coco):
    cat_map = {c["id"]: c["name"] for c in coco["categories"]}
    counts = Counter(cat_map[ann["category_id"]] for ann in coco["annotations"])
    return dict(sorted(counts.items(), key=lambda x: -x[1]))

def hash_images(split_dir):
    hashes = {}
    for fname in os.listdir(split_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            with open(os.path.join(split_dir, fname), "rb") as f:
                hashes[hashlib.md5(f.read()).hexdigest()] = fname
    return hashes

coco_data = {}
for split in ["train", "valid", "test"]:
    split_dir = f"braille_coco/{split}"
    coco = load_coco(split_dir)
    coco_data[split] = coco

    bad_imgs = check_images(split_dir)
    bad_boxes = check_boxes(coco)
    print(f"\n{split}: {len(coco['images'])} images, {len(coco['annotations'])} annotations")
    print(f"  corrupt images: {len(bad_imgs)} | invalid boxes: {len(bad_boxes)}")

print("\nTrain class distribution:")
for cls, n in class_distribution(coco_data["train"]).items():
    flag = "  <-- LOW (fewer than 10)" if n < 10 else ""
    print(f"  {cls}: {n}{flag}")

train_hashes = hash_images("braille_coco/train")
valid_hashes = hash_images("braille_coco/valid")
test_hashes = hash_images("braille_coco/test")
print("\nTrain/Valid duplicate images:", len(set(train_hashes) & set(valid_hashes)))
print("Train/Test duplicate images:", len(set(train_hashes) & set(test_hashes)))


train: 1823 images, 29018 annotations
  corrupt images: 0 | invalid boxes: 12

valid: 974 images, 15775 annotations
  corrupt images: 0 | invalid boxes: 0

test: 1093 images, 31166 annotations
  corrupt images: 0 | invalid boxes: 4

Train class distribution:
  E: 3170
  S: 2511
  O: 1976
  A: 1949
  L: 1774
  N: 1536
  D: 1462
  I: 1398
  R: 1376
  T: 1367
  H: 1322
  Q: 1285
  C: 846
  V: 757
  M: 695
  U: 668
  K: 648
  B: 642
  Y: 627
  G: 626
  P: 604
  F: 482
  X: 425
  W: 362
  Z: 257
  J: 253

Train/Valid duplicate images: 3
Train/Test duplicate images: 4


In [ ]:
def coco_to_yolo_labels(coco, labels_out_dir):
    os.makedirs(labels_out_dir, exist_ok=True)

    categories = sorted(coco["categories"], key=lambda c: c["id"])
    cat_id_to_yolo = {c["id"]: i for i, c in enumerate(categories)}
    class_names = [c["name"] for c in categories]

    images = {img["id"]: img for img in coco["images"]}
    anns_per_image = {}
    for ann in coco["annotations"]:
        anns_per_image.setdefault(ann["image_id"], []).append(ann)

    for img_id, img in images.items():
        w, h = img["width"], img["height"]
        base = os.path.splitext(img["file_name"])[0]
        lines = []
        for ann in anns_per_image.get(img_id, []):
            x, y, bw, bh = ann["bbox"]
            xc, yc = (x + bw / 2) / w, (y + bh / 2) / h
            nw, nh = bw / w, bh / h
            cls = cat_id_to_yolo[ann["category_id"]]
            lines.append(f"{cls} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")
        with open(os.path.join(labels_out_dir, base + ".txt"), "w") as f:
            f.write("\n".join(lines))

    return class_names

class_names = None
for split in ["train", "valid", "test"]:
    names = coco_to_yolo_labels(coco_data[split], f"braille_yolo/labels/{split}")
    class_names = names  # same across splits
    print(f"{split}: labels written")

print("Classes:", class_names)

train: labels written
valid: labels written
test: labels written
Classes: ['braille', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [ ]:
import shutil

for split in ["train", "valid", "test"]:
    out_dir = f"braille_yolo/images/{split}"
    os.makedirs(out_dir, exist_ok=True)
    for fname in os.listdir(f"braille_coco/{split}"):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            shutil.copy(f"braille_coco/{split}/{fname}", os.path.join(out_dir, fname))

# Verify images and labels line up
for split in ["train", "valid", "test"]:
    n_images = len([f for f in os.listdir(f"braille_yolo/images/{split}") if f.lower().endswith((".jpg",".jpeg",".png"))])
    n_labels = len([f for f in os.listdir(f"braille_yolo/labels/{split}") if f.endswith(".txt")])
    print(f"{split}: {n_images} images, {n_labels} labels")

train: 1822 images, 1823 labels
valid: 974 images, 974 labels
test: 1084 images, 1093 labels


In [ ]:
yaml_content = f"""train: images/train
val: images/valid
test: images/test

nc: {len(class_names)}
names: {class_names}
"""

with open("braille_yolo/data.yaml", "w") as f:
    f.write(yaml_content)

print(yaml_content)

train: images/train
val: images/valid
test: images/test

nc: 27
names: ['braille', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']



In [ ]:
!pip install ultralytics -q

from ultralytics import YOLO

# Clear any stale cache files from prior failed runs
for f in ["braille_yolo/labels/train.cache", "braille_yolo/labels/valid.cache", "braille_yolo/labels/test.cache"]:
    if os.path.exists(f):
        os.remove(f)

model = YOLO("yolov8n.pt")

results = model.train(
    data="braille_yolo/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    patience=20,
    fliplr=0.0  # disabled: flipping could turn one Braille letter into a different valid letter
)

Ultralytics 8.4.101 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=braille_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

In [ ]:
metrics = model.val(data="braille_yolo/data.yaml", split="test")

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.4.101 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,010,913 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1275.5±654.8 MB/s, size: 44.5 KB)
val: Scanning /content/braille_yolo/labels/test... 1084 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1084/1084 1.7Kit/s 0.6s
val: New cache created: /content/braille_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 68/68 3.7it/s 18.6s
                   all       1084      30939      0.842      0.773      0.845      0.589
                     A        785       2479      0.774      0.735      0.799      0.517
                     B        521        776      0.823      0.659      0.763      0.483
                     C        611       1034      0.774      0.838      0.863      0.602
                     D        681       1559      0.891      0.788      

In [ ]:
shutil.copy("runs/detect/train/weights/best.pt", "braille_yolov8_best.pt")

from google.colab import files
files.download("braille_yolov8_best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>